# Chapter 9. Spatial analysis and crime mapping

*Companion notebook to* **Computational Criminology: Research Methods, Ethics, and Reproducible Practice with R and Python** *by Dr Fahad Hameed Khan.*

Run the setup cell once, then run the code cell. Everything uses synthetic data, so nothing here describes a real person. The R version of this chapter is in `code/r/ch09_spatial.R`.

In [ ]:
# Setup: fetch the repository, install what this chapter needs,
# and build the synthetic data. Safe to run more than once.
import os
if not os.path.exists('computational-criminology'):
    !git clone -q https://github.com/drfahadhameedkhan/computational-criminology.git
%cd -q computational-criminology
!pip install -q numpy pandas matplotlib geopandas libpysal esda shapely
!python data/synthetic/make_synthetic_data.py

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("data/synthetic/burglary.csv")
OUTDIR = Path("figures/output")
OUTDIR.mkdir(parents=True, exist_ok=True)

try:
    import geopandas as gpd
    import matplotlib.pyplot as plt
    from libpysal.weights import Queen
    from esda.moran import Moran, Moran_Local
    from shapely.geometry import box
except ImportError as e:
    print("This chapter needs the geospatial stack. Install with:")
    print("    pip install geopandas libpysal esda matplotlib shapely")
    print("Missing:", e.name)
    raise SystemExit(0)

# --- book code: read a CSV of incidents and make spatial points ------------
crime = pd.read_csv(DATA)
pts = gpd.GeoDataFrame(
    crime,
    geometry=gpd.points_from_xy(crime.easting, crime.northing),
    crs=27700)                       # British National Grid: distances in metres

# --- build a 500 m grid and count points per cell --------------------------
xmin, ymin, xmax, ymax = pts.total_bounds
cell = 500.0
cols = np.arange(xmin, xmax + cell, cell)
rows = np.arange(ymin, ymax + cell, cell)
polys = [box(x, y, x + cell, y + cell) for x in cols[:-1] for y in rows[:-1]]
cells = gpd.GeoDataFrame(geometry=polys, crs=27700)
joined = gpd.sjoin(pts, cells, predicate="within", how="left")
counts = joined.groupby("index_right").size()
cells["count"] = cells.index.map(counts).fillna(0).astype(int)
cells = cells[cells["count"].notna()].reset_index(drop=True)

# --- book code: Moran's I with 999 permutations ---------------------------
w = Queen.from_dataframe(cells, use_index=False)   # neighbour relationships
w.transform = 'r'                                  # row-standardise the weights
mi = Moran(cells['count'].values, w, permutations=999)
print('Moran I =', round(mi.I, 3), ' pseudo p =', round(mi.p_sim, 3))

lm = Moran_Local(cells['count'].values, w, permutations=999)
# lm.q gives each cell's quadrant: 1 high-high, 3 low-low, etc.
sig = (lm.p_sim < 0.05) & (lm.q == 1)
print("high-high hot-spot cells (p<0.05):", int(sig.sum()))

# --- save a points + density figure (Figure 9.1 style) --------------------
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
pts.plot(ax=ax[0], markersize=3, color="#444")
ax[0].set_title("(a) incidents as points")
ax[0].set_aspect("equal"); ax[0].axis("off")
cells.plot(ax=ax[1], column="count", cmap="viridis", legend=True)
ax[1].set_title("(b) counts per 500 m cell")
ax[1].set_aspect("equal"); ax[1].axis("off")
fig.suptitle("Chapter 9: the same incidents as points and as a grid surface")
fig.tight_layout()
out = OUTDIR / "ch09_points_and_density.png"
fig.savefig(out, dpi=130)
print("figure saved to", out)

## Exercises

The exercises below assume no prior programming. Read the walkthrough first, then type the code yourself rather than copying it, because typing forces you to notice each step. Both tracks do the same job, so choose the language your department uses. R has the deeper tradition in spatial statistics; Python integrates more easily with general data work. Either is a sound first language for a criminologist.
Work through these in order. The later ones reward the habits the chapter argues for.
- Reproduce Figure 9.1 from any point dataset you can obtain, drawing the points and a density surface side by side. Change the bandwidth twice and write two sentences on how the map changes.
- Compute Moran's I for counts on a 500 m grid and again on a 1000 m grid. Report both, with permutation p-values, and explain any difference using the modifiable areal unit problem.
- Map the local Moran or Getis-Ord results and mark the significant hot spots. Apply a correction for multiple testing and state how many cells survive it.
- Take the busiest ten cells in one month and track them over the next three months. Discuss what regression to the mean implies for an intervention aimed at this month's worst places.
- Write a 300-word methods paragraph reporting your analysis to the standard set out in Chapter 16, naming every choice a reader would need to reproduce it.
These are for discussion and writing, and they do not need a computer.
- A council reports that half of all street robbery falls on two per cent of street segments and proposes saturation patrol there. Set out, as concentration and regression to the mean would, the case for and the case against.
- A predictive system trained on recorded crime sends patrols to the same districts each week. Explain the feedback loop, and propose one change to the data or the design that would weaken it.
- You are asked to map drug offences for a public report. Identify three display choices that would shape how readers judge the affected neighbourhoods, and state the choice you would defend.